<a href="https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice and rationale

My lane is Refresh / Content Opportunity Scoring. The goal is to prioritize content items for human review using observed search-performance signals.

I will start with **Logistic Regression** as a simple, interpretable classification method. The target is a binary decline proxy derived from the observed trend direction. The model will estimate the probability that a content item belongs to the declining class, allowing items to be ranked for review.

I will use only features that are available at the decision moment. I will exclude `trend_direction`, `trend_pct`, and `is_declining_label` from the feature set because they define or reveal the target. I will also exclude product flags and future-window information.

I will compare the model against my Week-4 rule-based baseline on the same evaluation rows using the same metric. If the model does not improve on the baseline, I will report that honestly. The model is intended for decision support, not to prove that refreshing content will cause performance to improve.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd

url = (
    "https://raw.githubusercontent.com/"
    "SoyebAhmed5/flyrank-internship-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will keep the evaluation data separate from the data used to train the model. Where the available fields allow it, I will use a time-aware split so that the model learns from earlier observations and is evaluated on later observations. I will avoid allowing the same content item or client to create unintended overlap between training and evaluation.

The split will be documented and used consistently for both Logistic Regression and the Week-4 baseline. I will fit preprocessing steps using the training data only. I will not use the sealed June 2026 test month to choose features, tune the model, or make development decisions.

This evaluation measures performance on the chosen split; it does not guarantee that the model will generalize to every client or future period.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import numpy as np

from sklearn.model_selection import GroupShuffleSplit

# Create the binary target from the observed trend direction.
df = df.copy()

df["target_declining"] = (
    df["trend_direction"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

# Metadata-only features: exclude the target and trend-derived fields.
numeric_features = [
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
]

feature_columns = numeric_features + categorical_features

# Keep rows with a usable target and client group.
model_df = df.dropna(
    subset=["trend_direction", "client_id"]
).copy()

X = model_df[feature_columns]
y = model_df["target_declining"]
groups = model_df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Evaluation rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Evaluation clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("\nTraining target rate:", round(y_train.mean(), 3))
print("Evaluation target rate:", round(y_test.mean(), 3))



Training rows: 23837
Evaluation rows: 6163
Training clients: 25
Evaluation clients: 7
Client overlap: 0

Training target rate: 0.55
Evaluation target rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model training and evaluation

I use Logistic Regression as an interpretable classification model. Numeric features are imputed with their training-set medians and standardized. Categorical features are imputed with their most frequent training-set values and one-hot encoded.

The model uses content metadata rather than recent performance windows that may overlap with the construction of the decline target. I evaluate ranking quality using Average Precision and also report ROC-AUC and threshold-based classification metrics.

All preprocessing is fitted on the training set only. The evaluation set remains held out until evaluation. I will compare this model with the Week-4 baseline on the same evaluation rows and metric.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

numeric_features = [
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

model_metrics = {
    "Average Precision": average_precision_score(y_test, y_prob),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "Precision at 0.5": precision_score(
        y_test, y_pred, zero_division=0
    ),
    "Recall at 0.5": recall_score(
        y_test, y_pred, zero_division=0
    ),
    "F1 at 0.5": f1_score(
        y_test, y_pred, zero_division=0
    ),
}

print("Logistic Regression — evaluation metrics")
display(pd.DataFrame(
    [{"Metric": k, "Value": v} for k, v in model_metrics.items()]
))

print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred))



Logistic Regression — evaluation metrics


,Metric,Value
0,Average Precision,0.542122
1,ROC-AUC,0.547755
2,Precision at 0.5,0.538937
3,Recall at 0.5,0.553827
4,F1 at 0.5,0.546280


Confusion matrix (rows = actual, columns = predicted):
[[1522 1492]
 [1405 1744]]


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The Logistic Regression model achieved an Average Precision of 0.5421 and ROC-AUC of 0.5478 on the held-out evaluation set. At a classification threshold of 0.5, precision was 0.5389, recall was 0.5538, and F1-score was 0.5463.

The confusion matrix shows that the model correctly identified 1,744 declining items and 1,522 non-declining items. However, it missed 1,405 declining items and incorrectly flagged 1,492 non-declining items. This indicates that the model has limited ability to distinguish declining from non-declining content using the selected metadata features alone.

These results should be interpreted alongside the Week-4 baseline before deciding whether the model provides an improvement. The evaluation uses a client-grouped split, so it measures generalization to clients excluded from training rather than future-time performance. The target is a proxy derived from the available trend direction field.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path

baseline_path = Path("work/outputs/baseline_action_score.csv")

if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
    print("Baseline shape:", baseline_df.shape)
    print("\nColumns:")
    print(baseline_df.columns.tolist())
    display(baseline_df.head())
else:
    print("Baseline CSV not found at:", baseline_path)
    print("Current directory:", Path.cwd())
    print("\nPossible baseline files:")
    for path in Path(".").rglob("baseline_action_score.csv"):
        print(path)


for pattern in ["*w04*baseline*.ipynb", "*baseline*.ipynb", "*baseline*.csv"]:
    matches = list(Path("/content").rglob(pattern))
    print(f"\n{pattern}")
    for path in matches[:20]:
        print(path)

Baseline CSV not found at: work/outputs/baseline_action_score.csv
Current directory: /content

Possible baseline files:

*w04*baseline*.ipynb

*baseline*.ipynb

*baseline*.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.